# Report Generation Agent

This notebook implements an example of a Report Generation Agent for single-table relational
data source, including a demo agent demo UI and evaluations with [Langfuse](https://langfuse.com/).

The data source implemented here is an [SQLite](https://sqlite.org/) database which is supported
natively by Python and saves the data in disk.
[SQLAlchemy](https://www.sqlalchemy.org/) is used as a SQL connection tool so this
SQL connection can be easily swapped for other databases.

The Report Generation Agent will provide an UI to read user queries in natural language
and procceed to make SQL queries to the database in order to produce the data for
the report. At the end, the Agent will provide a downloadable link to the report as
an `.xlsx` file.

This example also provides agent monitoring and evaluations using Langfuse.

## Setting up

The code below sets the notebook default folder, sets the default constants and checks the presence of the environment variables.

In [2]:
import os
import ssl
import urllib.request
import zipfile
from pathlib import Path

import certifi
import pandas as pd
from aieng.agent_evals.async_client_manager import AsyncClientManager

from implementations.report_generation.data.import_online_retail_data import import_online_retail_data


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "eval-agents":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

client_manager = AsyncClientManager.get_instance()
assert client_manager.configs.report_generation_db.database, (
    "[ERROR] The database path is not set! Please configure the REPORT_GENERATION_DB__DATABASE environment variable."
)

print("All environment variables have been set.")

DATA_FOLDER = Path("implementations/report_generation/data")
DATASET_PATH = DATA_FOLDER / "OnlineRetail.csv"

The notebook path has been set to: /Users/marcelolotif/workspace/eval-agents
All environment variables have been set.


## Dataset

The dataset used in this example is the
[Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail) dataset. It contains
information about invoices for products that were purchased by customers, which also includes
product quantity, the invoice date and the country that the custgomer resides in. For a more
detailed data structure, please check the [OnlineRetail.ddl](data/Online%20Retail.ddl) file.

## Downloading the Dataset

The code below will download and unzip the dataset to the `implementations/report_generation/data/` folder.

In [3]:
url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
zip_file_path = DATA_FOLDER / "online_retail.zip"
xlsx_file_path = DATA_FOLDER / "Online Retail.xlsx"

print("Downloading the dataset...")
ctx = ssl.create_default_context(cafile=certifi.where())
req = urllib.request.Request(url)
with urllib.request.urlopen(req, context=ctx) as resp, open(zip_file_path, "wb") as f:
    f.write(resp.read())

print("Extracting the dataset file...")
with zipfile.ZipFile(zip_file_path, "r") as zf:
    zf.extractall(DATA_FOLDER)

print("Converting the dataset file from .xls to .csv...")
df = pd.read_excel(xlsx_file_path)
df.to_csv(DATASET_PATH, index=False)

print("Done!")

Extracting the dataset file...
Converting the dataset file from .xls to .csv...
Done!


## Importing the Data

The code below will import the `.csv` dataset to the database at the path set by the `REPORT_GENERATION_DB__DATABASE` environment variable.

In [7]:
import_online_retail_data(DATASET_PATH)
print("Done!")

2026-02-17 13:43:51,794 INFO implementations.report_generation.data.import_online_retail_data: Creating tables according to the OnlineRetail.ddl file
2026-02-17 13:43:51,798 INFO implementations.report_generation.data.import_online_retail_data: Importing dataset from implementations/report_generation/data/OnlineRetail.csv to database at implementations/report_generation/data/OnlineRetail.db
2026-02-17 13:43:54,933 INFO implementations.report_generation.data.import_online_retail_data: Dataset imported successfully to database at implementations/report_generation/data/OnlineRetail.db


Done!
